In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
import sys
ROOT = '../../../'
sys.path.insert(0, ROOT)

import re
import numpy as np
import pandas as pd
import modules.measures as ms

rng = np.random.default_rng(0)

In [3]:
def _str_to_interval(s):
    m = re.match(r'\[([-\d.]+),\s*([-\d.inf]+)\)', str(s))
    if m:
        return pd.Interval(float(m.group(1)), float(m.group(2)), closed='left')
    return s

In [4]:
def _collapse_to_health_bracket(iv):
    """Map a 5-year age bracket to the matching health-survey bracket and the
    fraction of that bracket which falls inside it. Returns (target, weight)
    or None for brackets below 18 that have no health counterpart."""
    if iv == pd.Interval(15.0, 20.0, closed='left'):
        # health starts at 18, so only ages 18-19 of the [15, 20) span count
        return pd.Interval(18.0, 35.0, closed='left'), 2 / 5
    left = iv.left
    if left < 18:
        return None
    if left < 35:
        return pd.Interval(18.0, 35.0, closed='left'), 1.0
    if left < 50:
        return pd.Interval(35.0, 50.0, closed='left'), 1.0
    if left < 65:
        return pd.Interval(50.0, 65.0, closed='left'), 1.0
    return pd.Interval(65.0, np.inf, closed='left'), 1.0

In [5]:
data_age = pd.read_csv(ROOT + 'data/bayesian_network/age-fixed.csv')
data_health = pd.read_csv(ROOT + 'data/bayesian_network/health-fixed.csv')

data_age = data_age.rename(columns = {'gender': 'sex'})
data_age = data_age.set_index(['sex', 'age_group'])
data_age.index = data_age.index.set_levels(
    data_age.index.levels[data_age.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)

data_health = data_health.set_index(['sex', 'age_group'])
data_health.index = data_health.index.set_levels(
    data_health.index.levels[data_health.index.names.index('age_group')].map(_str_to_interval),
    level='age_group'
)
data_health = data_health.sort_index()

In [6]:
_mapped = [_collapse_to_health_bracket(iv) for iv in data_age.index.get_level_values('age_group')]
_keep = [m is not None for m in _mapped]

_weights = np.array([m[1] for m in _mapped if m is not None])
_target_age = [m[0] for m in _mapped if m is not None]
_sex = data_age.index.get_level_values('sex')[_keep]

data_age = (
    data_age[_keep]
    .mul(_weights, axis=0)
    .set_axis(pd.MultiIndex.from_arrays([_sex, _target_age], names=['sex', 'age_group']))
    .groupby(level=['sex', 'age_group'])
    .sum()
    .round()
    .astype(int)
    .sort_index()
)

# The Canada total is a derived sum, not an independent measurement: rounding each
# province column on its own can leave it off by 1 (only the [18, 35) bracket is
# fractional, via the 2/5 weight). Recompute it from the rounded provinces so it stays consistent.
data_age['Canada (excluding territories)'] = (
    data_age.drop(columns='Canada (excluding territories)').sum(axis=1)
)

---------------------------

In [7]:
age_provinces = data_age.iloc[:, :-1]
age_canada = data_age['Canada (excluding territories)']

# P(location) - Probability of being from a province
p_loc = (age_provinces / age_canada.sum()).sum()

# P(sex|location)
p_sex_given_loc = age_provinces.groupby('sex').sum() / age_provinces.sum()

# P(age | sex, loc)
p_age_given_sexloc = age_provinces.groupby('sex').apply(lambda g: g / g.sum())

# P(condition | sex, age)
p_cond_given_sexage = data_health.div(age_canada, axis=0)

# P(HBP | smoker, obese)
p_hbp_given_smoker_obese = pd.DataFrame(
    {False: [0.2, 0.3], True: [0.38, 0.55]}, # Obesity on columns
    index = [False, True]
)

In [8]:
p_hbp_given_smoker_obese

,False,True
False,0.2,0.38
True,0.3,0.55


In [9]:
def sample_single():
    # loc = rng.choice(p_loc.index, p=p_loc.values)
    loc = 'Alberta'

    sexes = p_sex_given_loc[loc]
    sex = rng.choice(p_sex_given_loc.index, p=sexes.values)

    ages = p_age_given_sexloc[loc][sex]
    age = rng.choice(ages.index.get_level_values(1), p=ages.values)

    conds = p_cond_given_sexage.loc[(sex, age)]
    obese      = rng.random() < conds['Obese']
    smoker     = rng.random() < conds['Current smoker']
    vaccinated = rng.random() < conds['Recently vaccinated']

    hbp = rng.random() < p_hbp_given_smoker_obese.loc[smoker, obese]

    return (loc, sex, age, obese, smoker, hbp, vaccinated)

In [10]:
id_data = pd.DataFrame(
    [sample_single() for _ in range(387_000)],
    columns = ['province', 'sex', 'age', 'obese', 'smoker', 'hbp', 'vaccinated']
)
id_data

,province,sex,age,obese,smoker,hbp,vaccinated
0,Alberta,Male,"[18.0, 35.0)",True,True,False,False
1,Alberta,Male,"[50.0, 65.0)",False,False,True,False
2,Alberta,Male,"[18.0, 35.0)",False,False,False,False
3,Alberta,Female,"[35.0, 50.0)",True,False,False,False
4,Alberta,Male,"[35.0, 50.0)",False,False,False,False
...,...,...,...,...,...,...,...
386995,Alberta,Female,"[35.0, 50.0)",True,False,True,False
386996,Alberta,Female,"[35.0, 50.0)",True,False,False,False
386997,Alberta,Male,"[65.0, inf)",False,False,False,True
386998,Alberta,Female,"[65.0, inf)",True,True,False,True


In [11]:
med_data = pd.read_csv(ROOT + 'data/calgary_kids.csv')
med_data = med_data.drop(columns='Unnamed: 0')
med_data

,address,age,sex,conditions,smoker,hbp
0,T3H,17,male,NaN,False,False
1,T2B,0,male,NaN,False,False
2,T1Y,2,female,Q56,False,False
3,T1Y,13,female,Q02,False,False
4,T3A,14,female,NaN,False,False
...,...,...,...,...,...,...
4995,T3E,9,female,NaN,False,True
4996,T3A,24,prefer not to say,NaN,False,False
4997,T3E,23,prefer not to say,NaN,False,False
4998,T2K,18,male,Q22,False,False


In [12]:
# --- Line up age representations -------------------------------------------
# Bin med_data's integer ages into the same left-closed brackets used in id_data.
age_bins = pd.IntervalIndex.from_breaks([18.0, 35.0, 50.0, 65.0, np.inf], closed='left')
med_data['age'] = pd.cut(med_data['age'], age_bins).astype(object)

# Keep only age brackets present in BOTH frames (drops med_data's under-18s,
# and any id_data bracket the medical data never reaches).
common_ages = set(med_data['age'].dropna()) & set(id_data['age'].dropna())
med_data = med_data[med_data['age'].isin(common_ages)].reset_index(drop=True)
id_data = id_data[id_data['age'].isin(common_ages)].reset_index(drop=True)

# --- Decapitalise sex labels in id_data ------------------------------------
id_data['sex'] = id_data['sex'].replace({'Male': 'male', 'Female': 'female'})

In [13]:
med_data

,address,age,sex,conditions,smoker,hbp
0,T2E,"[18.0, 35.0)",male,NaN,True,False
1,T2X,"[18.0, 35.0)",prefer not to say,Q34,False,False
2,T3K,"[18.0, 35.0)",male,NaN,False,False
3,T2P,"[18.0, 35.0)",female,Q30,False,False
4,T3C,"[18.0, 35.0)",male,NaN,False,False
...,...,...,...,...,...,...
1358,T3M,"[18.0, 35.0)",male,Q16,False,False
1359,T1S,"[18.0, 35.0)",female,Q92,False,False
1360,T3A,"[18.0, 35.0)",prefer not to say,NaN,False,False
1361,T3E,"[18.0, 35.0)",prefer not to say,NaN,False,False


In [14]:
id_data

,province,sex,age,obese,smoker,hbp,vaccinated
0,Alberta,male,"[18.0, 35.0)",True,True,False,False
1,Alberta,male,"[18.0, 35.0)",False,False,False,False
2,Alberta,female,"[18.0, 35.0)",False,False,True,True
3,Alberta,female,"[18.0, 35.0)",True,False,False,False
4,Alberta,male,"[18.0, 35.0)",True,False,False,False
...,...,...,...,...,...,...,...
114703,Alberta,male,"[18.0, 35.0)",True,False,False,False
114704,Alberta,male,"[18.0, 35.0)",True,False,False,True
114705,Alberta,male,"[18.0, 35.0)",False,False,False,False
114706,Alberta,male,"[18.0, 35.0)",False,False,False,False


In [15]:
ms.cross_entropy(med_data[['sex', 'smoker', 'hbp']], id_data[['sex', 'smoker', 'hbp']])

(np.float64(1.8639467086583288), np.float64(2.683057004455682))

In [21]:
ms.jiang(med_data[['sex', 'smoker', 'hbp']], id_data[['sex', 'smoker', 'hbp']])

np.float64(0.013763178337457674)

In [24]:
ms.entropy(med_data.iloc[:,1:])

np.float64(6.11333729054221)

In [25]:
data2 = pd.read_csv(ROOT + 'data/calgary_kids_recommendations.csv')
data2

,age,sex,conditions,smoker,hbp
0,"[15, 20)",male,NaN,False,False
1,"[0, 5)",male,NaN,False,False
2,"[0, 5)",female,Q56,False,False
3,"[10, 15)",female,Q02,False,False
4,"[10, 15)",female,NaN,False,False
...,...,...,...,...,...
4995,"[5, 10)",female,NaN,False,True
4996,"[20, 25)",prefer not to say,NaN,False,False
4997,"[20, 25)",prefer not to say,NaN,False,False
4998,"[15, 20)",male,Q22,False,False


In [26]:
ms.cross_entropy(med_data[['sex', 'smoker', 'hbp']], data2[['sex', 'smoker', 'hbp']])

(np.float64(2.295377482476176), np.float64(2.2572409866097862))

In [27]:
ms.jiang(med_data[['sex', 'smoker', 'hbp']], data2[['sex', 'smoker', 'hbp']])

np.float64(0.009255791714509442)

In [ ]:
ms.entropy(med_data.iloc[:,1:])